In [ ]:
# SECTION 1 — SETUP & PATHS
 
import os, sys, json, random
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from tqdm import tqdm
 

TRAIN_DIR = "/kaggle/input/datasets/nehareddysabbidi/dataset/wYe7pBJ7-train/train"
TEST_DIR  = "/kaggle/input/datasets/nehareddysabbidi/dataset/Pa7a3Hin-test-public/Pa7a3Hin-test-public"
OCR_WEIGHTS = "/kaggle/input/ocr-model/BEST_OCR.pth"        
SWINIR_WEIGHTS = "/kaggle/input/swinir-weights/003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x4_GAN.pth"
WORK_DIR  = "/kaggle/working"
 
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
 
# Sanity check dirs
print("TRAIN →", os.listdir(TRAIN_DIR)[:5])
print("TEST  →", os.listdir(TEST_DIR)[:5])
 
 

Device: cuda
TRAIN → ['Scenario-A', 'Scenario-B']
TEST  → ['track_18210', 'track_15070', 'track_21876', 'track_21569', 'track_16337']


In [ ]:
# SECTION 2 — CHARACTER MAP & CRNN MODEL
 
CHARS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
BLANK = 36                        # CTC blank index
 
ctoi = {c: i for i, c in enumerate(CHARS)}
itoc = {i: c for i, c in enumerate(CHARS)}
 
# map common noise characters to BLANK so encode never KeyErrors
for _ch in (" ", "-", ".", "_"):
    ctoi[_ch] = BLANK
 
 
def encode_text(text):
    text = text.upper()
    return [ctoi[c] for c in text if c in ctoi]
 
 
def decode_pred(seq):
    """CTC greedy decode — collapse repeats and strip blanks."""
    s, prev = "", -1
    for i in seq:
        if i != prev and i < len(CHARS):
            s += itoc.get(i, "")
        prev = i
    return s
 
 
class CRNN(nn.Module):
    def __init__(self, num_classes=36):
        super().__init__()
 
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2, 2)),                    # 32→16
 
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d((2, 2)),                    # 16→8
 
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(),
 
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d((2, 1)),                    # 8→4
 
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512), nn.ReLU(),
 
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d((2, 1)),                    # 4→2
        )
 
        # After CNN: feature map [B, 512, 2, W] → LSTM input = 512*2 = 1024
        self.rnn1 = nn.LSTM(1024, 256, bidirectional=True, batch_first=True)
        self.rnn2 = nn.LSTM(512,  256, bidirectional=True, batch_first=True)
 
        self.fc = nn.Linear(512, num_classes + 1)   # +1 for CTC blank
 
    def forward(self, x):
        f = self.cnn(x)                              # [B, 512, 2, W]
        b, c, h, w = f.size()
        f = f.permute(0, 3, 1, 2).reshape(b, w, c * h)   # [B, W, 1024]
        o, _ = self.rnn1(f)
        o, _ = self.rnn2(o)
        return self.fc(o)                            # [B, W, num_classes+1]
 

In [ ]:
# SECTION 3 — COLLECT ALL TRACKS & TRAIN/VAL SPLIT

 
all_tracks = []
for scenario in ["Scenario-A", "Scenario-B"]:
    for country in ["Brazilian", "Mercosur"]:
        p = os.path.join(TRAIN_DIR, scenario, country)
        all_tracks += [os.path.join(p, t) for t in os.listdir(p)]
 
random.seed(42)
random.shuffle(all_tracks)
 
split = int(0.9 * len(all_tracks))
train_t = all_tracks[:split]
val_t   = all_tracks[split:]
 
print(f"Total tracks : {len(all_tracks)}")
print(f"Train tracks : {len(train_t)}")
print(f"Val   tracks : {len(val_t)}")
 
 

Total tracks : 20000
Train tracks : 18000
Val   tracks : 2000


In [ ]:
# SECTION 4 — PHASE 1: TRAIN OCR ON HR IMAGES
 
class PlateDataset(Dataset):
    """HR images → OCR training."""
    def __init__(self, tracks):
        self.samples = []
        for tp in tracks:
            ann  = json.load(open(os.path.join(tp, "annotations.json")))
            text = ann["plate_text"]
            for i in range(1, 6):
                png = os.path.join(tp, f"hr-00{i}.png")
                jpg = os.path.join(tp, f"hr-00{i}.jpg")
                img = png if os.path.exists(png) else jpg
                self.samples.append((img, text))
 
    def __len__(self):
        return len(self.samples)
 
    def __getitem__(self, idx):
        img_p, label = self.samples[idx]
        img = cv2.imread(img_p)
        if img is None:
            raise ValueError(f"Image not found → {img_p}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (128, 32))
        img = torch.tensor(img).permute(2, 0, 1).float() / 255.0
        return img, label
 
 
ctc_loss_fn = nn.CTCLoss(blank=BLANK, zero_infinity=True)
 
 
def prepare_batch(labels, device):
    targets, lengths = [], []
    for t in labels:
        e = encode_text(t)
        targets += e
        lengths.append(len(e))
    return torch.tensor(targets).to(device), torch.tensor(lengths)
 
 
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    loop = tqdm(loader, leave=False)
    for imgs, labels in loop:
        imgs = imgs.to(device)
        out  = model(imgs)
 
        log_probs = out.log_softmax(2)
        b, w, _   = log_probs.shape
        input_lengths = torch.full((b,), w, dtype=torch.long)
 
        targets, target_lengths = prepare_batch(labels, device)
 
        loss = ctc_loss_fn(
            log_probs.permute(1, 0, 2),
            targets,
            input_lengths,
            target_lengths,
        )
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
        total_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")
 
    return total_loss / len(loader)
 
 
def val_accuracy_hr(model, val_tracks, device):
    """Evaluate on HR images (used during Phase 1)."""
    model.eval()
    correct, total = 0, 0
    for tp in tqdm(val_tracks, desc="Val-HR", leave=False):
        ann = json.load(open(os.path.join(tp, "annotations.json")))
        gt  = ann["plate_text"]
        preds = []
        for i in range(1, 6):
            png = os.path.join(tp, f"hr-00{i}.png")
            jpg = os.path.join(tp, f"hr-00{i}.jpg")
            p   = png if os.path.exists(png) else jpg
            img = cv2.imread(p)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (128, 32))
            t   = torch.tensor(img).permute(2, 0, 1).float() / 255.
            t   = t.unsqueeze(0).to(device)
            with torch.no_grad():
                seq = model(t).argmax(2)[0].cpu().numpy()
            preds.append(decode_pred(seq))
        final = Counter(preds).most_common(1)[0][0]
        if final == gt:
            correct += 1
        total += 1
    acc = 100 * correct / total
    print(f"  Val HR Accuracy : {acc:.2f}%")
    return acc
 
 

In [ ]:
# ---------- Run Phase 1 ----------
print("\n===== PHASE 1: Training on HR images =====")
 
train_loader = DataLoader(PlateDataset(train_t), batch_size=32, shuffle=True, num_workers=2)
 
model     = CRNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
 
EPOCHS       = 15
best_hr_acc  = 0.0
best_dst     = os.path.join(WORK_DIR, "BEST_OCR.pth")
 
for epoch in range(EPOCHS):
    avg_loss = train_one_epoch(model, train_loader, optimizer, device)
    print(f"Epoch [{epoch+1}/{EPOCHS}]  Loss: {avg_loss:.4f}")
 
    acc = val_accuracy_hr(model, val_t, device)
 
    if acc > best_hr_acc:
        best_hr_acc = acc
        torch.save(model.state_dict(), best_dst)
        print(f"  🚀 NEW BEST SAVED → BEST_OCR.pth  ({best_hr_acc:.2f}%)")
 
print(f"\nPhase-1 done. Best HR val accuracy: {best_hr_acc:.2f}%")
 
 
 


===== PHASE 1: Training on HR images =====


Epoch [1/15]  Loss: 0.9738


  Val HR Accuracy : 94.60%
  🚀 NEW BEST SAVED → BEST_OCR.pth  (94.60%)


Epoch [2/15]  Loss: 0.0253


  Val HR Accuracy : 95.65%
  🚀 NEW BEST SAVED → BEST_OCR.pth  (95.65%)


...

  Val HR Accuracy : 97.40%


Epoch [14/15]  Loss: 0.0026


  Val HR Accuracy : 97.40%


Epoch [15/15]  Loss: 0.0021


  Val HR Accuracy : 97.30%

Phase-1 done. Best HR val accuracy: 97.55%


In [ ]:
# SECTION 6 — PHASE 2: FINE-TUNE ON LR IMAGES
 
class LRFineTuneDataset(Dataset):
    """LR images with light augmentation."""
    def __init__(self, tracks):
        self.samples = []
        for tp in tracks:
            ann  = json.load(open(os.path.join(tp, "annotations.json")))
            text = ann["plate_text"]
            for i in range(1, 6):
                p1 = os.path.join(tp, f"lr-00{i}.png")
                p2 = os.path.join(tp, f"lr-00{i}.jpg")
                p  = p1 if os.path.exists(p1) else p2
                self.samples.append((p, text))
 
    def __len__(self):
        return len(self.samples)
 
    def __getitem__(self, idx):
        path, text = self.samples[idx]
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (128, 32))
 
        # Augmentation
        if random.random() < 0.5:
            img = cv2.GaussianBlur(img, (3, 3), 0)
        if random.random() < 0.3:
            img = img.astype("float32") + np.random.normal(0, 8, img.shape)
 
        img = np.clip(img, 0, 255).astype("float32") / 255.
        t   = torch.tensor(img).permute(2, 0, 1).float()
        return t, text
 
 
def val_accuracy_lr(model, val_tracks, device):
    """Evaluate on LR images (used during Phase 2)."""
    model.eval()
    correct, total = 0, 0
    for tp in tqdm(val_tracks, desc="Val-LR", leave=False):
        ann = json.load(open(os.path.join(tp, "annotations.json")))
        gt  = ann["plate_text"]
        preds = []
        for i in range(1, 6):
            p1 = os.path.join(tp, f"lr-00{i}.png")
            p2 = os.path.join(tp, f"lr-00{i}.jpg")
            p  = p1 if os.path.exists(p1) else p2
            img = cv2.imread(p)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (128, 32))
            t   = torch.tensor(img).permute(2, 0, 1).float() / 255.
            t   = t.unsqueeze(0).to(device)
            with torch.no_grad():
                seq = model(t).argmax(2)[0].cpu().numpy()
            preds.append(decode_pred(seq))
        final = Counter(preds).most_common(1)[0][0]
        if final == gt:
            correct += 1
        total += 1
    acc = 100 * correct / total
    print(f"  Val LR Accuracy : {acc:.2f}%")
    return acc
 
 
print("\n===== PHASE 2: Fine-tuning on LR images =====")
 
# Load best HR model as starting point
model = CRNN().to(device)
model.load_state_dict(torch.load(best_dst, map_location=device))
 
lr_train_loader = DataLoader(
    LRFineTuneDataset(train_t), batch_size=32, shuffle=True, num_workers=2
)
 
optimizer2 = torch.optim.Adam(model.parameters(), lr=1e-5)
 
best_lr_acc = 0.0
LR_EPOCHS   = 15
 
for epoch in range(LR_EPOCHS):
    # Training loop
    model.train()
    loop = tqdm(lr_train_loader, leave=False)
    for imgs, labels in loop:
        imgs = imgs.to(device)
        out  = model(imgs)
 
        log_probs = out.log_softmax(2)
        b, w, _   = log_probs.shape
        input_lengths = torch.full((b,), w, dtype=torch.long)
 
        targets, target_lengths = prepare_batch(labels, device)
 
        loss = ctc_loss_fn(
            log_probs.permute(1, 0, 2),
            targets,
            input_lengths,
            target_lengths,
        )
        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()
        loop.set_postfix(loss=f"{loss.item():.4f}")
 
    print(f"\nEpoch [{epoch+1}/{LR_EPOCHS}] — VALIDATION")
    acc = val_accuracy_lr(model, val_t, device)
 
    ckpt_path = os.path.join(WORK_DIR, f"LR_FINETUNE_{epoch}.pth")
    torch.save(model.state_dict(), ckpt_path)
 
    if acc > best_lr_acc:
        best_lr_acc = acc
        torch.save(model.state_dict(), os.path.join(WORK_DIR, "BEST_LR_OCR.pth"))
        print("  🚀 NEW BEST SAVED → BEST_LR_OCR.pth")
 
print(f"\nPhase-2 done. Best LR val accuracy: {best_lr_acc:.2f}%")
 
 



===== PHASE 2: Fine-tuning on LR images =====



Epoch [1/15] — VALIDATION


  Val LR Accuracy : 43.20%
  🚀 NEW BEST SAVED → BEST_LR_OCR.pth



Epoch [3/15] — VALIDATION


  Val LR Accuracy : 54.85%
  🚀 NEW BEST SAVED → BEST_LR_OCR.pth


...

  Val LR Accuracy : 62.25%



Epoch [14/15] — VALIDATION


  Val LR Accuracy : 62.25%



Epoch [15/15] — VALIDATION


  Val LR Accuracy : 62.85%
  🚀 NEW BEST SAVED → BEST_LR_OCR.pth

Phase-2 done. Best LR val accuracy: 62.85%


In [ ]:
#  SECTION 7 — LOAD BEST_LR_OCR & FULL TRAIN EVALUATION
 
print("\n===== Loading BEST_LR_OCR for final evaluation =====")
 
model = CRNN().to(device)
model.load_state_dict(
    torch.load(os.path.join(WORK_DIR, "BEST_LR_OCR.pth"), map_location=device)
)
model.eval()
print("BEST_LR_OCR loaded ✅")
 
 
def ocr_lr(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (128, 32))
    img = img.astype("float32") / 255.
    t   = torch.tensor(img).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        out  = model(t)
        prob = out.softmax(2)
        pred = prob.argmax(2)[0].cpu().numpy()
    return decode_pred(pred)
 
 
def predict_track(tp):
    ann = json.load(open(os.path.join(tp, "annotations.json")))
    gt  = ann["plate_text"]
    preds = []
    for i in range(1, 6):
        p1 = os.path.join(tp, f"lr-00{i}.png")
        p2 = os.path.join(tp, f"lr-00{i}.jpg")
        p  = p1 if os.path.exists(p1) else p2
        preds.append(ocr_lr(p))
    final = max(set(preds), key=preds.count)
    return final, gt
 
 
def full_train_accuracy(tracks):
    correct = 0
    for tp in tqdm(tracks):
        pred, gt = predict_track(tp)
        if pred == gt:
            correct += 1
    acc = 100 * correct / len(tracks)
    print("\n==============================")
    print("TOTAL TRACKS :", len(tracks))
    print("CORRECT      :", correct)
    print("LR ACCURACY  :", round(acc, 2), "%")
    print("==============================")
    return acc
 
 
full_train_accuracy(all_tracks)
 


===== Loading BEST_LR_OCR for final evaluation =====
BEST_LR_OCR loaded ✅


100%|██████████| 20000/20000 [10:22<00:00, 32.12it/s]


TOTAL TRACKS : 20000
CORRECT      : 16209
LR ACCURACY  : 81.05 %


81.045

In [15]:
full_train_accuracy(val_t)

100%|██████████| 2000/2000 [00:49<00:00, 40.57it/s]


TOTAL TRACKS : 2000
CORRECT      : 1281
LR ACCURACY  : 64.05 %


64.05

In [ ]:
# SECTION 8 — TEST SET PREDICTION → submission.zip
 
print("\n===== Generating test predictions =====")
 
test_tracks = sorted(os.listdir(TEST_DIR))
print(f"Test tracks : {len(test_tracks)}")
 
predictions = {}
 
for track in tqdm(test_tracks):
    tp = os.path.join(TEST_DIR, track)
 
    lr_files = sorted([
        f for f in os.listdir(tp)
        if ("lr" in f.lower()) and (f.endswith(".jpg") or f.endswith(".png"))
    ])
 
    if not lr_files:
        predictions[track] = ""
        continue
 
    preds = []
    for fname in lr_files:
        lr_path = os.path.join(tp, fname)
        txt = ocr_lr(lr_path)     # raw LR → resize 128×32 → CRNN → decode
        preds.append(txt)
 
    # Majority vote across all frames
    final = Counter(preds).most_common(1)[0][0]
    predictions[track] = final
 
# ---------- Write CSV ----------
csv_path = os.path.join(WORK_DIR, "submission.csv")
with open(csv_path, "w") as f:
    f.write("track_id,plate_text\n")
    for track, pred in predictions.items():
        f.write(f"{track},{pred}\n")
 
print(f"submission.csv written — {len(predictions)} rows")
 
# Preview first 10
with open(csv_path) as f:
    for line in f.readlines()[:11]:
        print(line.strip())
 
# ---------- Zip models + submission ----------
zip_path = os.path.join(WORK_DIR, "submission.zip")
os.system(
    f"cd {WORK_DIR} && zip submission.zip "
    f"submission.csv BEST_OCR.pth BEST_LR_OCR.pth"
)
print(f"\n✅ submission.zip ready → {zip_path}")
print("Contains: submission.csv | BEST_OCR.pth | BEST_LR_OCR.pth")
 
# ---------- Kaggle file link ----------
try:
    from IPython.display import FileLink, display
    display(FileLink("submission.zip"))
except Exception:
    pass
 
print("\n🎉 All done!")
 


===== Generating test predictions =====
Test tracks : 1000


100%|██████████| 1000/1000 [00:27<00:00, 36.45it/s]


submission.csv written — 1000 rows
track_id,plate_text
track_10005,FIO0729
track_10015,BAX0901
track_10016,BJD4645
track_10035,BBZ8740
track_10039,ATC7553
track_10058,AWM9327
track_10073,BBX0749
track_10088,AL2033
track_10089,BCI1840
track_10091,BBA6476
updating: submission.csv (deflated 60%)
updating: BEST_OCR.pth (deflated 8%)
updating: BEST_LR_OCR.pth (deflated 8%)

✅ submission.zip ready → /kaggle/working/submission.zip
Contains: submission.csv | BEST_OCR.pth | BEST_LR_OCR.pth


/kaggle/working/submission.zip


🎉 All done!
